# Detection & Classification using deep learning

ResNet18-style CNN classifier, selective search proposals.
Run from the repository root with `dataset/{train,val,test}` in place (see README).

In [ ]:
from src.cnn import Sign_Classifier, TrafficSignDataset
from src.crop import crop_images_from_folder
from src.detection import detection_images_in_folder
from src.evaluation import evaluate_detections, load_detections, load_ground_truth
from src.negatives import generate_negative_samples, get_negative_prediction

train_images, train_labels = "dataset/train/images/", "dataset/train/labels/"
val_images, val_labels = "dataset/val/images/", "dataset/val/labels/"
test_images = "dataset/test/images/"
train_path = "dataset/train/cropped_images/"
val_path = "dataset/val/cropped_images/"
model_path = "traffic_sign_resnet18.pth"

# The CNN is much more confident than the SVM, so it needs stricter thresholds.
# n_jobs=1: torch already uses every core.
detection_kwargs = dict(min_region_size=5000, sign_threshold=0.98, light_threshold=0.96, n_jobs=1)

## Cropping and negative examples generation

In [ ]:
crop_images_from_folder(train_images, train_labels, train_path)
crop_images_from_folder(val_images, val_labels, val_path)
generate_negative_samples(train_images, train_labels, train_path, target_sizes=[(200, 200), (400, 400)], num_samples=2)
generate_negative_samples(train_images, train_labels, train_path, target_sizes=[(128, 128), (64, 64)], num_samples=2)

## First model training

In [ ]:
val_dataset = TrafficSignDataset(val_path)
clf = Sign_Classifier().fit(TrafficSignDataset(train_path), val_dataset, save_path=model_path)

## Load a trained model (optional)

In [ ]:
clf = Sign_Classifier().load_model(model_path)

## Hard negative mining

Run detection on the **train** images and add every detection that overlaps no annotated box to the training set as `none`.

In [ ]:
detection_images_in_folder(train_images, clf, "detections_train.csv", output_folder=None, **detection_kwargs)
get_negative_prediction(load_detections("detections_train.csv"), load_ground_truth(train_labels), train_images, train_path)

## Second model training

In [ ]:
clf = Sign_Classifier().fit(TrafficSignDataset(train_path), val_dataset, save_path=model_path)

## Final detection on the validation set

In [ ]:
detection_images_in_folder(val_images, clf, "detections_val.csv", output_folder="output_images/val", **detection_kwargs)
report = evaluate_detections(load_detections("detections_val.csv"), load_ground_truth(val_labels))

## Detection on the test set

In [ ]:
detection_images_in_folder(test_images, clf, "detections_test.csv", output_folder="output_images/test", **detection_kwargs)